In [2]:
import pandas as pd
import torch

In [3]:
df=pd.read_csv("student_performance.csv")

In [4]:
df.head()

,student_id,weekly_self_study_hours,attendance_percentage,class_participation,total_score,grade
0,1,18.5,95.6,3.8,97.9,A
1,2,14.0,80.0,2.5,83.9,B
2,3,19.5,86.3,5.3,100.0,A
3,4,25.7,70.2,7.0,100.0,A
4,5,13.4,81.9,6.9,92.0,A


In [5]:
torch.manual_seed(42)


In [6]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [48]:
df["grade"].nunique()

5

In [9]:
from torch.utils.data import DataLoader,Dataset
from sklearn.model_selection  import train_test_split
import torch.nn as nn
import torch.optim as optim

In [98]:
x=df.iloc[:,1:-1].values
y=df.iloc[:,-1].values

In [99]:
x

array([[ 18.5,  95.6,   3.8,  97.9],
       [ 14. ,  80. ,   2.5,  83.9],
       [ 19.5,  86.3,   5.3, 100. ],
       ...,
       [ 15.2,  72.5,   6.8,  85.9],
       [  5.9,  79.8,   7.5,  72.4],
       [  8. ,   nan,   nan,   nan]])

In [100]:
y

array(['A', 'B', 'A', ..., 'A', 'B', nan], dtype=object)

In [101]:
class custom(Dataset):
  def __init__(self,features,labels):
    self.features=torch.tensor(features,dtype=torch.float32)
    self.labels=torch.tensor(labels,dtype=torch.long)

  def __getitem__(self, index) :
    return self.features[index],self.labels[index]
  def __len__(self):
    return len(self.features)


In [102]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
y=le.fit_transform(y)

In [103]:
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()


In [104]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [105]:
x_train=sc.fit_transform(x_train)
x_test=sc.transform(x_test)

In [106]:
x_train.shape,y_train.shape

((510224, 4), (510224,))

In [107]:
train_dataset = custom(x_train, y_train)
test_dataset = custom(x_test, y_test)

In [108]:
train_df=DataLoader(train_dataset,batch_size=32,shuffle=True,pin_memory=True)
val_df=DataLoader(test_dataset,batch_size=32,shuffle=False,pin_memory=True)

In [109]:
val_df

In [110]:
from torch.nn.modules.linear import Linear
class mynn(nn.Module):
  def __init__(self,n_featuers):
    super().__init__()
    self.model=nn.Sequential(
        nn.Linear(n_featuers,128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(128,64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(64,5),


    )
  def forward(self,x):
      return self.model(x)


In [85]:
model=mynn(x_train.shape[1])
model.to(device)
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=0.001,weight_decay=1e-4)


In [86]:
epochs=10


In [87]:
from tqdm import tqdm

In [88]:
!pip install torchmetrics

In [89]:
from torchmetrics import Accuracy

In [94]:
metric = Accuracy(task="multiclass", num_classes=5).to(device)

In [91]:
model.train()
for epoch in range(epochs):
  total_loss=0
  for batch_features,batch_labels in tqdm(train_df):
    batch_features,batch_labels=batch_features.to(device),batch_labels.to(device)
    output=model(batch_features)
    loss=criterion(output,batch_labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_loss+=loss.item()
  print(f"epoch{epoch+1}:Loss:{total_loss/len(train_df)}")


100%|██████████| 15945/15945 [01:01<00:00, 258.04it/s]


epoch1:Loss:0.27974020616691964


100%|██████████| 15945/15945 [00:40<00:00, 391.33it/s]


epoch2:Loss:0.24448857674028815


100%|██████████| 15945/15945 [00:54<00:00, 294.97it/s]


...

epoch8:Loss:0.21900933689039753


100%|██████████| 15945/15945 [00:40<00:00, 391.12it/s]


epoch9:Loss:0.2194602461715745


100%|██████████| 15945/15945 [00:41<00:00, 386.33it/s]

epoch10:Loss:0.2166402064184385


In [92]:
model.eval()

mynn(
  (model): Sequential(
    (0): Linear(in_features=4, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=64, out_features=5, bias=True)
  )
)

In [112]:
correct=0
metric.reset()
total=0
with torch.no_grad():
  for batch_features, batch_labels in tqdm(val_df):

            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)
            output = model(batch_features)
            predicted = torch.argmax(output, dim=1)
            metric.update(output, batch_labels)
            correct += (predicted == batch_labels).sum().item()
            total += batch_labels.size(0)
accuracy = correct / total
print(accuracy)
print(metric.compute())


100%|██████████| 3987/3987 [00:06<00:00, 644.58it/s]

0.9691351249647214
tensor(0.9691, device='cuda:0')


In [113]:
!pip install torchinfo

In [114]:
from torchinfo import summary
summary(model ,input_size=x_train.shape)

Layer (type:depth-idx)                   Output Shape              Param #
mynn                                     [510224, 5]               --
├─Sequential: 1-1                        [510224, 5]               --
│    └─Linear: 2-1                       [510224, 128]             640
│    └─BatchNorm1d: 2-2                  [510224, 128]             256
│    └─ReLU: 2-3                         [510224, 128]             --
│    └─Dropout: 2-4                      [510224, 128]             --
│    └─Linear: 2-5                       [510224, 64]              8,256
│    └─BatchNorm1d: 2-6                  [510224, 64]              128
│    └─ReLU: 2-7                         [510224, 64]              --
│    └─Dropout: 2-8                      [510224, 64]              --
│    └─Linear: 2-9                       [510224, 5]               325
Total params: 9,605
Trainable params: 9,605
Non-trainable params: 0
Total mult-adds (Units.GIGABYTES): 4.90
Input size (MB): 8.16
Forward/backward p

In [116]:
x_test.shape

(127556, 4)

In [117]:
x_train.shape

(510224, 4)